# 04 – Wind Radius / Core Size

Analyses the radial extent of 10 m winds for Hurricane Kirk across all
OIFS SST-perturbation experiments, following the wind_radius approach in
Ribberink et al. (2025).

**Method**: For each timestep, cut an E–W and N–S slice through the storm
centre (from the storm track) in the 10 m wind speed field. Plot the
wind speed contours in storm-relative coordinates. Key thresholds:
- 17 m/s (tropical storm force)
- 34 m/s (hurricane force)
- 50 m/s (~Category 3)

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')
sys.path.insert(0, '../ribberink_code')

import numpy as np
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from oifs_adapter import RUNS, OIFSRun
from run_track import run_name_to_safe
import hurricane_functions as hf

## 1. Load pre-computed tracks

In [ ]:
tracks = {}
for run_name in RUNS:
    safe_name = run_name.replace('+', 'p').replace('-', 'm')
    path = f'../plots/tracks/track_{safe_name}.nc'
    if os.path.exists(path):
        tracks[run_name] = xr.open_dataset(path)
        print(f'Loaded: {run_name}')

## 2. Compute wind radius slices for each run

In [ ]:
wind_r = {}

for run_name, run_dir in RUNS.items():
    if run_name not in tracks:
        print(f'Skipping {run_name}: no track')
        continue
    print(f'Wind radius slices: {run_name}...')

    oifs_run = OIFSRun(run_dir)
    wind_ds = oifs_run.get_10m_wind()
    track_ds = tracks[run_name]

    # Compute wind speed magnitude
    ws = hf.magnitude(wind_ds['u10m'], wind_ds['v10m'])

    wind_r[run_name] = {}

    for j in range(len(track_ds.time)):
        lat_j = float(track_ds.lat[j])
        lon_j = float(track_ds.lon[j])

        wj = ws.isel(time=j)

        # Handle reduced Gaussian fields provided as flattened 'values' points
        # by rebuilding a 2D (lat, lon) grid for robust nearest-point slicing.
        if 'values' in wj.dims and 'lat' in wj.coords and 'lon' in wj.coords:
            wj = (
                wj.to_dataset(name='ws')
                .set_index(values=['lat', 'lon'])
                .unstack('values')['ws']
                .sortby('lat')
                .sortby('lon')
            )

        lon_vals = wj.lon.values
        lon_is_360 = float(np.nanmax(lon_vals)) > 180.0

        # Convert track longitude to match grid convention (0..360 or -180..180)
        if lon_is_360 and lon_j < 0:
            lon_target = (lon_j + 360.0) % 360.0
        elif (not lon_is_360) and lon_j > 180:
            lon_target = lon_j - 360.0
        else:
            lon_target = lon_j

        lat_idx = int(np.abs(wj.lat.values - lat_j).argmin())
        lon_idx = int(np.abs(wj.lon.values - lon_target).argmin())

        # E-W slice (constant lat)
        r_EW = wj.isel(lat=lat_idx)
        r_EW = r_EW.assign_coords(lon=r_EW.lon - lon_target)  # storm-relative

        # N-S slice (constant lon)
        r_NS = wj.isel(lon=lon_idx)
        r_NS = r_NS.assign_coords(lat=r_NS.lat - lat_j)  # storm-relative

        if j == 0:
            wind_r[run_name]['EW'] = r_EW
            wind_r[run_name]['NS'] = r_NS
        else:
            wind_r[run_name]['EW'] = xr.concat(
                [wind_r[run_name]['EW'], r_EW], dim='time', join='outer'
            )
            wind_r[run_name]['NS'] = xr.concat(
                [wind_r[run_name]['NS'], r_NS], dim='time', join='outer'
            )

    print(f'  Done.')

## 4. R17 radius time series (core size metric)

In [ ]:
# Compute R17 (radius of 17 m/s wind) for each timestep by finding max extent
# where wind speed >= 17 m/s in the E-W slice

# Reset matplotlib to default state to eliminate any inherited font configurations
plt.rcdefaults()

# Load ET-transition start times from timing analysis summary
import pandas as pd
ett_summary_path = '../plots/tracks/ett_start_summary.csv'
ett_start_times = {}
if os.path.exists(ett_summary_path):
    ett_df = pd.read_csv(ett_summary_path)
    for _, row in ett_df.iterrows():
        run_name = str(row.get('run', ''))
        et_start = row.get('et_start_time_utc', '')
        if run_name and isinstance(et_start, str) and et_start.strip():
            ett_start_times[run_name] = np.datetime64(et_start)

fig, ax = plt.subplots(figsize=(10, 6))

for run_name, res in wind_r.items():
    ew = res['EW']
    r17_list = []
    for t in range(len(ew.time)):
        ws_slice = ew.isel(time=t)
        lons = ew.lon.values
        ws_vals = ws_slice.values
        # Find all lons where wind >= 17 m/s
        above = np.where(ws_vals >= 17)[0]
        if len(above) > 0:
            r17 = (lons[above.max()] - lons[above.min()]) / 2 * 111  # degrees to km (approx)
        else:
            r17 = 0.0
        r17_list.append(r17)

    t_vals = np.array(ew.time.values)
    r17_vals = np.array(r17_list)
    c = colors_line.get(run_name, 'k')

    ax.plot(t_vals, r17_vals, '.-',
            color=c, label=run_name,
            clip_on=(run_name != '+3K'))  # allow +3K final value to overshoot

    # Larger marker for ETT start time (same run color)
    et_t = ett_start_times.get(run_name)
    if et_t is not None and len(t_vals) > 0:
        idx_et = int(np.abs(t_vals.astype('datetime64[ns]') - et_t.astype('datetime64[ns]')).argmin())
        ax.plot(t_vals[idx_et], r17_vals[idx_et], 'o', ms=12, color=c,
                markeredgecolor='k', markeredgewidth=0.8, zorder=6,
                clip_on=(run_name != '+3K'))

ax.set_xlabel('Date')
ax.set_ylabel('R17 (km, from E-W slice)')
ax.set_title('Hurricane Kirk – Tropical Storm Wind Radius (R17)')
ax.set_ylim(0, 3000)
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
plt.savefig('../plots/kirk_R17.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_R17.png')

## 5. Direction-aware core-radius metrics (batch output)

Loads the CSV outputs produced by `scripts/core_radius_metrics.py` and plots
storm-motion directional radii (forward/backward/left/right) plus mean radius.

Expected files:
- `../plots/tracks/core_radius_baserun.csv`
- `../plots/tracks/core_radius_m3K.csv`
- `../plots/tracks/core_radius_p3K.csv`
- `../plots/tracks/core_radius_p6K.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

tracks_dir = Path("../plots/tracks")
runs = {
    "Control": "core_radius_baserun.csv",
    "-3K": "core_radius_m3K.csv",
    "+3K": "core_radius_p3K.csv",
    "+6K": "core_radius_p6K.csv",
}

frames = {}
for label, fn in runs.items():
    p = tracks_dir / fn
    if p.exists():
        frames[label] = pd.read_csv(p)
    else:
        print(f"Missing file: {p}")

if not frames:
    raise FileNotFoundError("No directional core-radius CSV files found in ../plots/tracks")

thresholds = [17, 34, 50]
for thr in thresholds:
    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    for label, df in frames.items():
        t = pd.to_datetime(df["time"], errors="coerce")

        axes[0].plot(t, df[f"r{thr}_mean_km"], label=f"{label} mean", linewidth=2)

        # Directional asymmetry diagnostics.
        fb = df[f"r{thr}_forward_km"] - df[f"r{thr}_backward_km"]
        lr = df[f"r{thr}_left_km"] - df[f"r{thr}_right_km"]
        axes[1].plot(t, fb, label=f"{label} F-B", linestyle="-")
        axes[1].plot(t, lr, label=f"{label} L-R", linestyle="--")

    axes[0].set_title(f"R{thr} mean radius over time")
    axes[0].set_ylabel("Radius (km)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(ncol=2)

    axes[1].set_title(f"R{thr} directional asymmetry")
    axes[1].set_ylabel("Difference (km)")
    axes[1].set_xlabel("Time")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(ncol=4)

    plt.tight_layout()
    plt.show()

# Optional single-run directional panel (edit selected_run as needed)
selected_run = "Control"
if selected_run in frames:
    df = frames[selected_run]
    t = pd.to_datetime(df["time"], errors="coerce")

    fig, axs = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
    for i, thr in enumerate([17, 34, 50]):
        axs[i].plot(t, df[f"r{thr}_forward_km"], label="forward")
        axs[i].plot(t, df[f"r{thr}_backward_km"], label="backward")
        axs[i].plot(t, df[f"r{thr}_left_km"], label="left")
        axs[i].plot(t, df[f"r{thr}_right_km"], label="right")
        axs[i].plot(t, df[f"r{thr}_mean_km"], label="mean", linewidth=2, color="black")
        axs[i].set_title(f"{selected_run}: R{thr} directional radii")
        axs[i].set_ylabel("Radius (km)")
        axs[i].grid(True, alpha=0.3)
        axs[i].legend(ncol=5)

    axs[-1].set_xlabel("Time")
    plt.tight_layout()
    plt.show()